# Pooled Bayesian IDM

In [ ]:
import arviz as az
import numpy as np
"""
# If pymc3
"""
# import pymc3 as pm
# from theano import tensor as tt
"""
# If pymc4
"""
import pymc as pm
import aesara.tensor as tt

import random
import os
import sys
import matplotlib.pyplot as plt
plt.rcParams['text.usetex'] = True

import pickle

from os import path

import warnings
warnings.filterwarnings("ignore")

np.random.seed(1116)

class Config:
    dt = 0.2
    

In [ ]:
# "leader_id": leader_id,
# "leader_tidx": leader_tidx,
# "follower_id": follower_id,
# "follower_tidx": follower_tidx,
# "time": common_times,
# "leader_x": leader_x_interp,
# "leader_y": leader_y_interp,
# "follower_x": follower_x,
# "follower_y": follower_y,
# "gap": gap,
# "leader_speed": leader_speed_interp,
# "follower_speed": follower_speed,
# "relative_speed": relative_speed
    
def load_training_data(step, pair_id_list):
    # Load the data
    with open('./filtered_car_following_pairs.pkl', 'rb') as f:
        tracks = pickle.load(f)
    for pair_id in pair_id_list:
        xt_temp = tracks[pair_id]['follower_x'][:-1]
        vt_temp = tracks[pair_id]['follower_speed'][:-1]
        s_temp = tracks[pair_id]['gap'][:-1] 
        dv_temp = tracks[pair_id]['relative_speed'][:-1]
        label_x_temp = tracks[pair_id]['follower_x'][1:]
        label_v_temp = tracks[pair_id]['follower_speed'][1:]
        if pair_id == pair_id_list[0]:
            xt = xt_temp
            vt = vt_temp
            s = s_temp
            dv = dv_temp
            label_x = label_x_temp
            label_v = label_v_temp
        else:
            xt = np.concatenate((xt,xt_temp))
            vt = np.concatenate((vt,vt_temp))
            s = np.concatenate((s,s_temp))
            dv = np.concatenate((dv, dv_temp))
            label_x = np.concatenate((label_x, label_x_temp))
            label_v = np.concatenate((label_v, label_v_temp))
    if step!=1:
        xt = xt[::step]
        vt = vt[::step]
        s = s[::step]
        dv = dv[::step]
        label_x = label_x[::step]
        label_v = label_v[::step]
    
    print("ID list:", pair_id_list, ", Data size:", label_v.shape)
    return xt, vt, s, dv, label_x, label_v

def Bayesian_IDM_pool(base_path):
    xt, vt, s, dv, label_x, label_v = load_training_data(step=5, pair_id_list=range(500))
    
    print("training size:", label_v.shape[0])
    
    labels = np.vstack((label_v, label_x))
    dt = Config.dt

    model = pm.Model()

    D = 5
    
    with model:
        def IDM_xv(VMAX, DSAFE, TSAFE, AMAX, AMIN, DELTA, s, vt, dv, xt):
            sn = DSAFE + vt * TSAFE + vt * dv / (2 * tt.sqrt(AMAX * AMIN))
            a = AMAX * (1 - (vt / VMAX) ** DELTA - (sn / s) ** 2)
            return tt.stack([vt + a * Config.dt, xt + vt * Config.dt + 0.5 * a * Config.dt**2])
        
        mu_prior = pm.floatX(np.array([0,0,0,0,0]))
        parameters_normalized = pm.MvNormal('mu_normalized', mu_prior, chol=np.eye(D))
        
        log_parameters = pm.Deterministic('log_mu', parameters_normalized*np.array([.1, .1, .1, .1, .1])
                                      +np.array([2., 0.69, 0.47, .4, .51]))
        parameters = pm.Deterministic('mu', tt.exp(log_parameters))
        
        DELTA = 4
        
        s_a = pm.Exponential('s_a', lam=5e3) # 5e3/5e4
        s_v = 0.0046
        s_x = 0.027
        
        covs = tt.stacklists([[(s_a ** 2 * Config.dt ** 2 + s_v ** 2), 0.5 * s_a ** 2 * Config.dt ** 3],
                                  [0.5 * s_a ** 2 * Config.dt ** 3, (s_a ** 2 * Config.dt ** 4) / 4 + s_x ** 2]])
        
        mu = IDM_xv(parameters[0], parameters[1], parameters[2], parameters[3],
                    parameters[4], DELTA, s, vt, dv, xt)
        
        v_obs = pm.MvNormal('obs', mu=mu.T, cov=covs, observed=labels.T)

        tr = pm.sample(2500, tune=3000, random_seed=16, init='jitter+adapt_diag_grad', chains=1,
                       cores=14, discard_tuned_samples=True, return_inferencedata=True, target_accept=0.90)
    return tr, model

In [ ]:
base_path = ' '
tr, model = Bayesian_IDM_pool(base_path)

In [ ]:
import pickle
from pickle import UnpicklingError

cache = "./Bayesian_IDM_pooled_I24-Motion.pkl"
if path.exists(cache):
    try:
        fp = open(cache, 'rb')
        tr = pickle.load(fp)
        fp.close()
        print("Load trace", cache, ": done!")
    except UnpicklingError:
        os.remove(cache)
        print('Removed broken cache:', cache)
else:
    output_file = open(cache, 'wb')
    pickle.dump(tr, output_file)
    output_file.close()
    print("Generated and Saved", output_file, ": done!")

In [ ]:
_ = az.plot_trace(tr, var_names=["mu"], coords={"mu_dim_0":0}, compact=True)
_ = az.plot_trace(tr, var_names=["mu"], coords={"mu_dim_0":1}, compact=True)
_ = az.plot_trace(tr, var_names=["mu"], coords={"mu_dim_0":2}, compact=True)
_ = az.plot_trace(tr, var_names=["mu"], coords={"mu_dim_0":3}, compact=True)
_ = az.plot_trace(tr, var_names=["mu"], coords={"mu_dim_0":4}, compact=True)
_ = az.plot_trace(tr, var_names=["s_a"], compact=True)

In [ ]:
az.summary(tr, var_names=["mu","log_mu","s_a"])

In [ ]:
import corner

import matplotlib

label_list = [r'$v_0$',r'$s_0$', r'$T$', r'$\alpha$', r'$\beta$']
fontsize = 18
matplotlib.rc('xtick', labelsize=fontsize) 
matplotlib.rc('ytick', labelsize=fontsize) 

figure = corner.corner(
    tr,
    var_names=['mu'],
    smooth=1.8,
    color = 'k',
    plot_contours=True,
    plot_density=True,
    plot_datapoints = False,
    bins = 20,
    show_titles=True,
    labels=label_list,
    reverse=False,
)

ax_new = figure.add_axes([.66, .66, .27, .27])
cov = np.cov(tr.posterior.mu[0,:,:], rowvar=False)
Dinv = np.diag(1 / np.sqrt(np.diag(cov)))
corr = Dinv @ cov @ Dinv
kwargs = {'cmap':'plasma','interpolation':'nearest', 'vmin':-1}
corr_show = ax_new.matshow(corr, **kwargs)
c_bar = figure.colorbar(corr_show, ax=ax_new, extend='both')
ax_new.set_xticklabels(['']+label_list)
ax_new.set_yticklabels(['']+label_list)
ax_new.set_title('Correlation Matrix')
for item in ([ax_new.title, ax_new.xaxis.label, ax_new.yaxis.label] +
             ax_new.get_xticklabels() + ax_new.get_yticklabels()):
    item.set_fontsize(fontsize)

# figure.savefig('../Figs/B_IDM_pool_car.pdf', dpi=300)